In [15]:
import os

print(os.environ.get("JAVA_HOME"))

/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home


In [19]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").appName("Dataops-copilot").getOrCreate()

In [ ]:
df = spark.read.parquet("../data/yellow_tripdata_2026-01.parquet")

In [24]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [25]:
df.select("fare_amount", "trip_distance").printSchema()

root
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)



In [28]:
df.select(
    "fare_amount",
    "trip_distance",
).summary("count", "min", "max", "mean").show()

+-------+-----------------+-----------------+
|summary|      fare_amount|    trip_distance|
+-------+-----------------+-----------------+
|  count|          3724889|          3724889|
|    min|          -2555.2|              0.0|
|    max|           2555.2|        269097.48|
|   mean|20.80425389321187|6.455646860884686|
+-------+-----------------+-----------------+



In [40]:
import pyspark.sql.functions as F

summary = df.selectExpr(
    "count(*) as row_count",
    "coalesce(avg(int(fare_amount IS NULL)), 0.0) as fare_null_rate",
    "coalesce(avg(int(trip_distance IS NULL)), 0.0) as trip_distance_null_rate ",
    "coalesce(avg(int(fare_amount < 0)), 0.0) as negative_fare_rate",
    "coalesce(avg(int(trip_distance <= 0)), 0.0) as invalid_trip_distance_rate",
)

summary.show()

+---------+--------------+-----------------------+--------------------+--------------------------+
|row_count|fare_null_rate|trip_distance_null_rate|  negative_fare_rate|invalid_trip_distance_rate|
+---------+--------------+-----------------------+--------------------+--------------------------+
|  3724889|           0.0|                    0.0|0.010594409658918695|       0.03375617367390008|
+---------+--------------+-----------------------+--------------------+--------------------------+



In [42]:
duplicates_all = df.groupBy(df.columns).count().filter(F.col("count") > 1)

In [ ]:
duplicates_all = df.groupBy(df.columns).count().filter(F.col("count") > 1)
total_rows = df.count()
duplicate_count = total_rows - duplicates_all.count()
print(f"Total Duplicate Rows: {duplicate_count}")

Total Duplicate Rows: 3724889
